# Solutions · Chapter 05-12 · The applied checkpoint

Worked answers to every exercise in `notebooks/05_regression/05-12_applied_checkpoint.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

frame = fetch_california_housing(as_frame=True).frame
FEATURES = [column for column in frame.columns if column != "MedHouseVal"]
TARGET = "MedHouseVal"
CEILING = frame[TARGET].max()

train, test = train_test_split(frame, test_size=0.2, random_state=0)
fit, watch = train_test_split(train, test_size=0.25, random_state=0)


def rmse(model, part):
    return float(np.sqrt(((part[TARGET] - model.predict(part[FEATURES])) ** 2).mean()))


chosen = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.1, random_state=0).fit(
    fit[FEATURES], fit[TARGET])

scored = test.copy()
scored["prediction"] = chosen.predict(test[FEATURES])
scored["error"] = scored["prediction"] - scored[TARGET]
on_ceiling = (scored[TARGET] >= CEILING - 1e-9).to_numpy()


def by_segment(column):
    out = []
    for name, group in scored.groupby(column, observed=True):
        out.append({str(column): name, "rows": len(group),
                    "RMSE": float(np.sqrt((group["error"] ** 2).mean())),
                    "mean error (+ = too high)": float(group["error"].mean()),
                    "mean actual": float(group[TARGET].mean())})
    return pd.DataFrame(out)


print("test RMSE %.4f on %d rows, %d of them at the ceiling"
      % (rmse(chosen, test), len(test), on_ceiling.sum()))

## Quick understanding

### E1 · The test score came out better than the watch score

**Explanation one: noise.** These are two samples of 4,128 rows each. The difference is 0.0131 RMSE,
which is far smaller than the spread you would see by reshuffling the split - and the *expected*
direction is the other way, so a small reversal is unremarkable.

**Explanation two: the watch set is slightly harder.** It happens to hold 204 capped rows against the
test set's 181 - 4.94% against 4.38% - and this chapter established that capped rows carry roughly four
times the squared error of the rest. That alone accounts for part of the gap.

**Which needs more evidence to rule out:** the second. Noise is the default and needs no support;
"the watch set is systematically harder" is a claim about the split and can be checked - repeat the whole
procedure over several random seeds and see whether the sign of the difference persists. **If it does, the
split is not exchangeable and something is wrong with how it was made.**

### E2 · Why the split came before the histograms

Because **anything you learn from looking is a decision you then make using that data.** Had we plotted the
target first, noticed the ceiling, and chosen a model or a transformation because of it, the test set would
have contributed to that choice - and its score would then measure how well we fitted the test set, not how
well the model generalises.

**The subtle version, which is the one that catches people:** it does not require you to *fit* on the test
rows. Merely *choosing* on them is enough. 04-05's rule is about information flow, not about `.fit()`.

### E3 · What each part is for

- **`fit`** - the only rows the model's parameters are allowed to be estimated from.
- **`watch`** - the only rows any *choice* is scored on. Which model, which depth, how many rounds, which
  features. Used as many times as you like.
- **`test`** - scored once, after every choice is frozen, to produce the number that is reported. Used to
  change nothing.

**The failure that follows from breaking each:** fitting on `watch` makes model selection optimistic;
choosing on `test` makes the reported number optimistic; and looking at `test` at all - even a histogram -
starts the second failure quietly.

## Hand calculation

### E4 · 450 rows at RMSE 0.40, 50 at RMSE 1.20

Square first, because **RMSE does not average - MSE does.**

$$\text{MSE}_{\text{all}} = \frac{450 \times 0.40^2 + 50 \times 1.20^2}{500}
= \frac{450 \times 0.16 + 50 \times 1.44}{500} = \frac{72 + 72}{500} = 0.288$$

$$\text{RMSE} = \sqrt{0.288} = \mathbf{0.5367}$$

**The arithmetic contains a coincidence worth noticing: both groups contribute exactly 72.** Fifty rows at
three times the error carry as much weight in the total as four hundred and fifty ordinary ones, because
squaring turns a factor of 3 into a factor of 9. **A tenth of the data, half the error.**

The wrong answer is `(450 x 0.40 + 50 x 1.20) / 500 = 0.48`, which averages the RMSEs directly. It is
always too low, and by more the more unequal the groups are.

### E5 · How bad must 10 rows be to move 500?

Total squared error needed for an overall RMSE of 0.50 across 500 rows:

$$500 \times 0.50^2 = 125$$

The 490 ordinary rows supply $490 \times 0.16 = 78.4$, so the remaining ten must supply $125 - 78.4 = 46.6$:

$$\text{MSE}_{\text{segment}} = \frac{46.6}{10} = 4.66, \qquad
\text{RMSE}_{\text{segment}} = \sqrt{4.66} = \mathbf{2.1587}$$

**Those ten rows have to be five and a half times worse than everything else just to move the headline by
0.10.** That is the sensitivity of a global metric to a small segment, in one number - and it is why the
capped rows in this chapter, at twice the error and 4.4% of the data, moved the overall RMSE by only
0.0318.

### E6 · Mean error +0.30, RMSE 0.50

The identity is $\text{MSE} = \text{bias}^2 + \text{variance}$:

$$0.50^2 = 0.30^2 + \sigma^2 \quad\Longrightarrow\quad \sigma^2 = 0.25 - 0.09 = 0.16
\quad\Longrightarrow\quad \sigma = \mathbf{0.40}$$

**And the reason this decomposition earns its place in a segment report:** a segment with bias 0.30 and
spread 0.40 needs a different fix from one with bias 0.00 and spread 0.50, even though their RMSEs are
similar. The first is being systematically mis-called and something is missing from the model; the second
is just noisy.

### E7 · Three rows, one at the ceiling

Errors: $2.1 - 1.9 = +0.2$, $3.4 - 3.9 = -0.5$, $5.0 - 5.0 = 0.0$.

**With all three rows:**
$$\sqrt{\frac{0.04 + 0.25 + 0.00}{3}} = \sqrt{0.096667} = \mathbf{0.3109}$$

**Without the capped row:**
$$\sqrt{\frac{0.04 + 0.25}{2}} = \sqrt{0.145} = \mathbf{0.3808}$$

**Report the first, and say the third row is censored.** Dropping it makes the number *look worse* here,
which is a useful accident: it shows that excluding awkward rows is not a trick that always flatters you.

The deeper answer is that neither number is trustworthy on its own. The capped row scored a perfect zero
**because the model happened to predict exactly the ceiling**, and the true value could have been anything
at or above 5.0. **That zero is not a success; it is an unknown recorded as a success**, and averaging it
in pretends we know something we do not.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

# --- E4: where the total squared error actually comes from ----------------
groups = [("450 ordinary rows\nRMSE 0.40", 450, 0.40, "#b0b0b0"),
          ("50 bad rows\nRMSE 1.20", 50, 1.20, "#c0392b")]
contributions = [count * value ** 2 for _, count, value, _ in groups]
shares = [100 * count / 500 for _, count, _, _ in groups]

positions = [0, 1]
left.bar([p - 0.2 for p in positions], shares, 0.36,
         color=[c for *_, c in groups], alpha=0.45, label="share of the rows")
left.bar([p + 0.2 for p in positions], [100 * c / sum(contributions) for c in contributions], 0.36,
         color=[c for *_, c in groups], label="share of the total squared error")
for p, share, contribution in zip(positions, shares, contributions):
    left.text(p - 0.2, share + 1.5, "%.0f%%" % share, ha="center", fontsize=9)
    left.text(p + 0.2, 100 * contribution / sum(contributions) + 1.5,
              "%.0f%%" % (100 * contribution / sum(contributions)), ha="center", fontsize=9)
left.set_xticks(positions)
left.set_xticklabels([name for name, *_ in groups], fontsize=9)
left.set_ylim(0, 105)
left.set_ylabel("per cent")
left.set_title("E4 - 10% of the rows, 50% of the error")
left.legend(fontsize=9)

# --- E5: how bad a segment must be, as a function of how small it is ------
sizes = np.arange(5, 260)
required = np.sqrt((500 * 0.50 ** 2 - (500 - sizes) * 0.40 ** 2) / sizes)
right.plot(sizes, required, lw=2.4, color="#2c5f9e")
right.axhline(0.40, color="#333333", lw=1.2, ls="--")
right.text(250, 0.47, "the error everywhere else", fontsize=9, color="#333333", ha="right")
right.scatter([10], [np.sqrt((500 * 0.25 - 490 * 0.16) / 10)], s=70, color="#c0392b", zorder=5)
right.annotate("E5: 10 rows must reach 2.16", xy=(10, 2.1587), xytext=(60, 2.35), fontsize=9.5,
               color="#c0392b",
               bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none"),
               arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.2))
right.set_xlabel("size of the bad segment (out of 500 rows)")
right.set_ylabel("RMSE the segment must reach")
right.set_title("E5 - to drag the overall RMSE from 0.40 to 0.50")

fig.suptitle("Why a global metric barely notices a small segment", y=1.02)
fig.tight_layout()
plt.show()

**Left: E4 as two bars.** The bad rows are a tenth of the data and half of the squared error. Squaring is
what does that - and it is the reason RMSE is the metric that *does* notice concentrated failures, more
than MAE would.

**Right: E5 generalised, and this is the figure to remember.** The curve is what a segment's RMSE has to
reach to move the headline from 0.40 to 0.50, plotted against how big the segment is. **It climbs without
limit as the segment shrinks.** At 200 rows a segment need only reach 0.6205; at 100 rows, 0.7810; at 50
rows, 1.0296; at 25 rows, 1.4000; at 10 rows, 2.1587; at 5 rows, 3.0265 - more than seven times the error
of everything around it, to shift one decimal place of the headline.

> **The practical reading: a global metric is a detector with a size threshold.** Anything smaller than a
> few per cent of your data is invisible to it no matter how badly it fails. That is not a flaw in RMSE -
> it is arithmetic - and it is the entire justification for segment reporting.

## Coding

### E8 · The worked one, restated

The pattern is three lines and never changes: **name the group, group by it, report three numbers.**
Reporting RMSE alone is the mistake the shape is designed to prevent - `rows` tells you whether to believe
it, and `mean error` tells you which direction the model is wrong in.

### E9 · Block groups with more than 10 rooms per household

In [ ]:
scored["many rooms"] = np.where(scored["AveRooms"] > 10, "AveRooms above 10", "ordinary")
print(by_segment("many rooms").to_string(index=False, float_format=lambda v: "%.4f" % v))

**45 rows at RMSE 0.6247 against 0.4552 for everything else - about 37% worse, with essentially no bias
(+0.0057).**

**No bias plus a larger spread is a different diagnosis from the capped rows**, which had a large bias.
Here the model is not systematically wrong about these areas, it is *less certain* about them - which is
what you would expect from a region of the feature space with 45 training-scale examples in it and unusual
values in every column.

**And 45 rows is small enough that the honest sentence is "worse, probably, and we cannot say by how
much".** Compare with the ceiling segment's 181 rows, a clear direction and a known mechanism.

### E10 · The most populous tenth

In [ ]:
threshold = scored["Population"].quantile(0.9)
scored["crowded"] = np.where(scored["Population"] >= threshold,
                             "top 10% by population", "the other 90%")
print("threshold: %.0f people in the block group\n" % threshold)
print(by_segment("crowded").to_string(index=False, float_format=lambda v: "%.4f" % v))

**0.4671 against 0.4563. Nothing.**

414 rows is a large enough segment to trust, the difference is 0.01 RMSE, and the biases are both within
0.01 of zero. **This segmentation finds no problem, and that is a result worth writing down.**

It is worth being clear about why, because "we checked and found nothing" is the most under-reported
finding in applied work: the value of a null segment is that it *closes* a question. Somebody will
eventually ask whether the model does worse in dense areas. The answer is on record, with the numbers.

### E11 · Which feature hides the biggest gap

In [ ]:
def worst_segment(column, bins=5):
    # cut into equal-sized bins, score each, return the worst and the best
    binned = pd.qcut(scored[column], bins, duplicates="drop")
    scores = []
    for name, group in scored.groupby(binned, observed=True):
        scores.append((float(np.sqrt((group["error"] ** 2).mean())), len(group), name))
    scores.sort()
    return {"feature": column, "worst bin RMSE": scores[-1][0], "best bin RMSE": scores[0][0],
            "gap": scores[-1][0] - scores[0][0], "rows in the worst bin": scores[-1][1]}


gaps = pd.DataFrame([worst_segment(column) for column in FEATURES])
print(gaps.sort_values("gap", ascending=False).to_string(index=False,
                                                         float_format=lambda v: "%.4f" % v))

**`AveOccup` finds the biggest gap - 0.6368 in its worst fifth against 0.3565 in its best, a spread of
0.2802 - and `MedInc` finds the smallest, 0.0514.**

**That ordering is the opposite of what most people would guess**, and it is the most useful thing in this
exercise. Median income is the strongest single predictor in the dataset, so it feels like the natural
thing to segment on - and it is the one column where the model performs most evenly. The columns that
expose the model are the ones describing *unusual household structure*, where the training data is thin
and the averages stop meaning what their names say.

**A caution about the method itself.** Every bin here holds around 826 rows because `qcut` makes them equal
in size, which is what makes the comparison fair. Cut on fixed thresholds instead and the extreme bins go
tiny, their RMSEs go noisy, and this table will confidently point you at whichever feature happens to have
the smallest tail bin. **Equal-sized bins, or the ranking is an artefact.**

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.4))
ordered = gaps.sort_values("gap", ascending=True)
positions = np.arange(len(ordered))

ax.barh(positions, ordered["worst bin RMSE"], 0.62, color="#c0392b", label="worst fifth")
ax.barh(positions, ordered["best bin RMSE"], 0.62, color="#1f6f4a", label="best fifth")
for y, (worst, best) in enumerate(zip(ordered["worst bin RMSE"], ordered["best bin RMSE"])):
    ax.text(worst + 0.008, y, "%.4f" % worst, va="center", fontsize=8.5, color="#c0392b")
    ax.text(best - 0.008, y, "%.4f" % best, va="center", ha="right", fontsize=8.5, color="white")
ax.axvline(rmse(chosen, test), color="#333333", lw=1.2, ls=":")
ax.set_ylim(-0.7, len(ordered) + 0.1)
ax.text(rmse(chosen, test) + 0.006, len(ordered) - 0.35, "overall %.4f" % rmse(chosen, test),
        fontsize=9, color="#333333")
ax.set_yticks(positions)
ax.set_yticklabels(ordered["feature"])
ax.set_xlim(0, 0.72)
ax.set_xlabel("RMSE on the test set")
ax.set_title("Split the test set five ways on each column: how far apart do the best and worst fifths get?")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
plt.show()

**Read it from the bottom up, and the surprise is which end `MedInc` is at.**

The bars are sorted by the gap between a column's best and worst fifth. **`MedInc` - the strongest
predictor in the dataset - is at the narrow end**, and `AveOccup`, `AveRooms` and `Population` are at the
wide end. Predictive strength and error evenness are simply different properties, and a column can have
plenty of one and none of the other.

**The one to act on is `AveOccup`.** Its worst fifth scores 0.6368 against an overall 0.4574, on 826 rows -
big enough to believe, and traceable to a mechanism the chapter already identified: block groups where
"households" is small or strange, so every per-household average is unreliable as an input.

### E12 · Refitting on fit and watch together

In [ ]:
both = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.1, random_state=0).fit(
    train[FEATURES], train[TARGET])

print("fitted on 'fit' only   (%5d rows):  test RMSE %.4f" % (len(fit), rmse(chosen, test)))
print("fitted on fit + watch  (%5d rows):  test RMSE %.4f" % (len(train), rmse(both, test)))
print("\nimprovement from 33%% more training data: %.4f"
      % (rmse(chosen, test) - rmse(both, test)))

**0.4574 to 0.4406 - a real improvement, from nothing but 4,128 extra rows.**

**Why you would expect it:** 05-08's learning curve. The held-out error of a flexible model keeps falling
with training size long after the training error has flattened, and 12,384 rows is not obviously enough
for a model with 400 boosting rounds over eight features. More data reduces variance, and variance is what
this model has.

**Why it is standard practice:** once the hyperparameters are chosen, the validation set has done its job.
Refitting the *chosen* configuration on all the training data is the normal final step - `GridSearchCV`
does it for you with `refit=True`.

**And why doing it in the order shown here would be cheating.** We fitted on fit+watch and then scored on
test, and the score improved - so we now report 0.4406. But the decision to refit was taken *after* seeing
0.4574, which makes the improvement one we went looking for. **The honest procedure is to decide the whole
recipe - including "refit on everything at the end" - before opening the test set**, and then run it once.

> The tell is simple and worth carrying: **if you would not have made this change had the number gone the
> other way, it is not a method, it is a search.** Doing the refit is correct. Doing it because the first
> test score disappointed is not.

## Interpretation

### E13 · Why is the oldest housing band worse?

**Explanation one: it is small.** 445 rows against 1,466 in the 31-45 band, so its RMSE is the noisiest of
the four and some of the 0.1449 gap is sampling variation.

**Explanation two: it holds more censored rows.** Older housing is concentrated in older, more expensive
coastal neighbourhoods, so the 46-52 band should contain a higher share of rows at the value ceiling - and
the chapter established what those rows do to an error.

**What to compute:** the capped share in each band, and each band's RMSE with the capped rows removed. If
explanation two is right, the gap should largely disappear.

In [ ]:
scored["age band"] = pd.cut(scored["HouseAge"], [0, 15, 30, 45, 52],
                            labels=["1-15", "16-30", "31-45", "46-52"])
rows = []
for name, group in scored.groupby("age band", observed=True):
    without = group[group[TARGET] < CEILING - 1e-9]
    rows.append({"age band": name, "rows": len(group),
                 "share at the ceiling": float((group[TARGET] >= CEILING - 1e-9).mean()),
                 "RMSE": float(np.sqrt((group["error"] ** 2).mean())),
                 "RMSE without capped rows": float(np.sqrt((without["error"] ** 2).mean()))})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: "%.4f" % v))

**The prediction was right and the explanation is still refuted, which is the interesting outcome.**

The capped share does rise steadily with age - 1.98%, 3.72%, 5.12%, 7.87% - exactly as explanation two
predicted. But removing those rows **widens** the gap rather than closing it. The oldest band goes from
0.5543 to 0.5379 while the 31-45 band goes from 0.4094 to 0.3849, so the distance between them moves from
**0.1449 to 0.1530.** Every band improves when its capped rows are dropped; the oldest one improves least.

**The clincher is the 31-45 band itself**, which carries 5.12% capped rows - more than either younger
band - and has the *lowest* RMSE of all four. If censoring drove the ranking, that could not happen.

**So the honest answer is: neither explanation is sufficient, and something else is going on.** Which is
the correct place to stop, and the next thing to look at is `HouseAge` itself.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.4))

counts = train["HouseAge"].value_counts().sort_index()
left.bar(counts.index, counts.values, width=0.85, color="#b0b0b0")
left.bar([52], [counts.loc[52]], width=0.85, color="#c0392b")
left.annotate("%d rows at exactly 52,\nagainst %d at 51"
              % (counts.loc[52], counts.loc[51]), xy=(52, counts.loc[52] * 0.85),
              xytext=(33, counts.loc[52] * 0.78), fontsize=10, color="#c0392b", ha="center",
              bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none"),
              arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.3))
left.set_xlabel("HouseAge, in years")
left.set_ylabel("training rows")
left.set_title("the same recording rule, on a different column")

table = pd.DataFrame(rows)
positions = np.arange(len(table))
left_bars = right.bar(positions - 0.19, table["RMSE"], 0.36, color="#c0392b", label="all rows")
right_bars = right.bar(positions + 0.19, table["RMSE without capped rows"], 0.36,
                       color="#1f6f4a", label="capped rows removed")
for x, (a, b) in enumerate(zip(table["RMSE"], table["RMSE without capped rows"])):
    right.text(x - 0.19, a + 0.006, "%.4f" % a, ha="center", fontsize=8, color="#c0392b")
    right.text(x + 0.19, b + 0.006, "%.4f" % b, ha="center", fontsize=8, color="#1f6f4a")
right.set_xticks(positions)
right.set_xticklabels(table["age band"])
right.set_ylim(0, 0.65)
right.set_xlabel("age of the housing, in years")
right.set_ylabel("RMSE on the test set")
right.set_title("removing the capped rows barely moves the ranking")
right.legend(fontsize=9)

fig.suptitle("E13 - HouseAge is censored too, and it is still not the explanation", y=1.02)
fig.tight_layout()
plt.show()

**`HouseAge` has a ceiling of its own.** 1,025 training rows sit at exactly 52 years against 41 at 51 -
the identical signature the target had, on a different column. Anything older than 52 was recorded as 52.

**So the 46-52 band is not "the oldest housing". It is "housing aged 46 to 52, plus everything older than
that, of unknown age".** A model given a feature that has been flattened at the top has less to work with
in exactly that region - and that is a far better candidate explanation than either of the two the
exercise offered.

> **The transferable habit: when one column turns out to be censored, check the others.** Whatever process
> capped the value at 500,001 dollars was applied by the same people to the same file, and it capped the
> age at 52 as well. Censoring is rarely a property of one column; it is a property of a data collection.

### E14 · A prediction of 5.4737, above the ceiling

**Possible, and not a bug.** The model was never told that 5.00001 is a maximum. It fits a smooth-ish
function of eight inputs, and for a block group whose features look even more expensive than the capped
ones, it extrapolates upwards. Boosting adds together tree outputs; nothing in that sum is clipped.

**44 of the 4,128 test predictions land above the ceiling**, and they are arguably the model's most
*useful* outputs: they identify areas the recording rule erased. A prediction of 5.47 is the model saying
"this one is well past the limit", which is information the target column cannot express.

**Whether you keep it depends on the contract.** If the deliverable is "predict the recorded value",
clipping predictions at 5.00001 is correct and will lower your RMSE, because no actual value exceeds it. If
the deliverable is "estimate what the area is worth", clipping destroys the only signal you had about the
top of the market. **The metric prefers clipping; the question may not.** Say which you did.

### E15 · The overall mean error is +0.0079, so the model is unbiased

**Averaging over the whole test set is what makes the claim look true.** The two rows of the ceiling table
are +0.0335 on 3,947 ordinary rows and -0.5509 on 181 capped ones. Those cancel:

$$\frac{3947 \times 0.0335 + 181 \times (-0.5509)}{4128} \approx 0.0079$$

**A near-zero global bias is compatible with severe, opposite biases in subgroups** - and this is the
default outcome, not an unlucky one, because a squared-error model is fitted to make the average residual
approximately zero. **You should expect the global bias to be near zero. It carries almost no information.**

The claim that *would* mean something is "no segment we can name has a bias materially different from
zero", and that requires naming segments and checking them. Here, one does.

## Debugging

### E16 · `train_test_split` in a loop, best result kept

**They are reporting the maximum of a distribution and calling it a mean.**

With no `random_state`, each pass draws a different split and produces a different test score by chance
alone. Keeping the best over - say - fifty passes reports the luckiest split of fifty, which is an
estimate of "how well could this model score if the test set were as favourable as possible", not "how
well will it do on new data".

**This is 04-07's problem wearing a different hat.** The test set was used to make a choice (which split
to report), so it stopped being a test set at the first iteration of the loop.

**The fix:** fix the seed, split once, report that. If you want to know how much a score varies across
splits, that is a legitimate and different question - run the loop and report the **mean and spread of all
of them**, never the best.

### E17 · Scaling fitted on the whole frame before splitting

**No, it is not harmless - but on this data it is nearly harmless, and the distinction matters.**

`StandardScaler` learns a mean and a standard deviation. Fitting it on `frame` means those two numbers were
computed using the test rows, so information crossed the boundary. That is leakage by definition.

**Why the effect here is tiny:** a mean and a standard deviation over 20,640 rows barely move when 4,128
of them are removed, so the transform is almost identical either way. And the chosen model is
tree-based - trees only ever compare values, so scaling changes nothing at all for it. 05-10's point.

**Why "tiny" is the wrong thing to conclude.** The same mistake with `SelectKBest`, a target encoder, or
any imputation that uses the target is not tiny at all - and the habit that produces one produces the
others. **The correct response is to fix the pipeline, not to measure how much this instance cost.** Put
the scaler inside a `Pipeline` and it becomes structurally impossible.

## Exam and interview reasoning

### E18 · "Your model has an RMSE of 0.46. Is that good?"

> "It depends on three things and I would want all of them before answering.
>
> **Against what baseline?** Predicting the training mean gives 1.16 here, so 0.46 is a 60% reduction -
> that is the number that says the model is doing something. Linear regression gives 0.75, so the extra
> complexity is buying a further 0.29.
>
> **In what units?** 0.46 is 46,000 dollars on values averaging 207,000 - roughly 22%. Whether that is
> good depends entirely on what the prediction is for.
>
> **Good for whom?** The overall figure hides a segment at 0.91 where the model is systematically 55,000
> dollars low, because those values were capped in the data. If the use case touches expensive coastal
> areas, 0.46 is the wrong number to be quoting at all."

**What is being tested:** whether you reach for a baseline unprompted, whether you can convert to units
someone can act on, and whether you volunteer the failure without being asked.

### E19 · "How would you know if your model was failing for a particular group?"

> "I would not know from the headline metric, so I would segment - by the variables someone would actually
> act on, and by the ones where the model has least data. For each segment I would report size, error and
> **direction** of the error, because a segment with a bias is a different problem from a segment that is
> merely noisy."

**"And if the group were 1% of the data?"**

> "Then no global metric will ever surface it - a segment that small has to be roughly five to ten times
> worse than everything else to move the headline by a tenth, so it can fail badly and invisibly. I would
> have to be told the group matters and go and measure it deliberately.
>
> With 1% I would also be careful about the measurement itself. On 4,000 test rows that is 40 rows, which
> is enough to notice a large effect and not enough to size it. I would report it with an interval, and if
> the group genuinely matters I would over-sample it into the evaluation set - which is allowed, as long as
> you do not then quote the pooled number as if it were representative."

## Transfer to a different situation

### E20 · A readmission model with AUC 0.82

**Segments I would insist on, and they are chosen by who is harmed:** age bands; the primary-diagnosis
groups; patients with and without a prior admission; the small-volume wards; and - as a data question
rather than a fairness one - patients whose records are incomplete, because missingness is usually
informative in hospital data.

**For each, the same three columns as this chapter:** how many patients, the error, and the direction.
Direction is the one that matters clinically. A model that under-predicts risk for a group sends those
patients home; one that over-predicts wastes beds. **Those are not the same failure and one metric cannot
distinguish them.**

**If a segment is small and bad**, the size makes the estimate uncertain, not the problem unimportant.
Report it as an interval, gather more evaluation data for that group specifically, and in the meantime say
so in the deployment notes. **What you must not do is let "the sample is too small to be sure" quietly
become "we found nothing"** - those are opposite conclusions and they get confused constantly.

### E21 · Delivery times recorded as "90+"

**This is the chapter's ceiling, in a different domain**, and the failures are the same three.

1. **Systematic under-prediction of the worst deliveries.** Trained on a target that stops at 90, the model
   will never predict 140, so every genuinely catastrophic delivery is called at roughly 90. The customers
   most affected are the ones the estimate fails hardest.
2. **A metric that looks fine**, because those orders are a small share of the rows.
3. **Confident nonsense on the tail**, if anyone uses the predictions for the exact case they most care
   about - the late ones.

**Two ways to handle it, and they answer different questions.**

- **Reframe as classification.** Predict `P(over 90 minutes)` instead of a duration. The target is not
  censored - every row's *label* is known exactly - so the problem disappears rather than being modelled
  around. This is usually what the business wanted anyway.
- **Model it as censored data.** Survival analysis exists for exactly this; a Tweedie or quantile loss also
  behaves better than squared error on a bounded target. More machinery, and it gives you an estimate of
  the *true* duration rather than the recorded one.

**And the cheap thing to do first:** find out whether the raw times still exist somewhere upstream. Very
often the censoring happened in a reporting layer and the real numbers are in a log. **Recovering the data
beats every modelling response to not having it.**

## Explain it to someone non-technical

### E22 · Why not to use it for expensive neighbourhoods

> When this data was collected, anyone recording a neighbourhood worth more than half a million dollars
> just wrote down "half a million". A neighbourhood worth 600,000 and one worth 1.2 million look identical
> in our file.
>
> The model learned from that file, so it has never seen the difference either. For ordinary
> neighbourhoods it is reliable - typically within about 20% of the real figure. For the expensive ones it
> is not just less accurate, it is **too low, nearly every time**, by about 55,000 dollars on average.
>
> So treat any high estimate as "at least this much" rather than a valuation. If the expensive
> neighbourhoods are the ones you care about, we need different data - no amount of work on the model will
> fix this.

*(122 words.)* The load-bearing sentence is "worth 600,000 and worth 1.2 million look identical in our
file", because it explains the cause rather than the symptom - and it makes clear that the fix is data
rather than effort.

## Optional challenge

### E23 · Modelling the ceiling explicitly

Two models: one that predicts *whether* a block group is at the ceiling, and one that predicts the value
for those below it. Combine them by weighting with the first model's probability.

In [ ]:
is_capped = (fit[TARGET] >= CEILING - 1e-9).astype(int)
gate = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=0).fit(
    fit[FEATURES], is_capped)

below = fit[fit[TARGET] < CEILING - 1e-9]
value = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.1, random_state=0).fit(
    below[FEATURES], below[TARGET])

chance_capped = gate.predict_proba(test[FEATURES])[:, 1]
combined = chance_capped * CEILING + (1 - chance_capped) * value.predict(test[FEATURES])
combined_error = combined - test[TARGET].to_numpy()


def score(errors):
    return float(np.sqrt((errors ** 2).mean()))


print(pd.DataFrame([
    {"rows": "all %d" % len(test), "one model": score(scored["error"].to_numpy()),
     "two-part model": score(combined_error)},
    {"rows": "the %d ordinary" % (~on_ceiling).sum(),
     "one model": score(scored["error"].to_numpy()[~on_ceiling]),
     "two-part model": score(combined_error[~on_ceiling])},
    {"rows": "the %d capped" % on_ceiling.sum(),
     "one model": score(scored["error"].to_numpy()[on_ceiling]),
     "two-part model": score(combined_error[on_ceiling])},
]).to_string(index=False, float_format=lambda v: "%.4f" % v))

print("\nhow well the gate alone spots a capped row - test AUC %.4f"
      % roc_auc_score(on_ceiling.astype(int), chance_capped))
flagged = chance_capped > 0.5
print("flagging at 0.5 catches %d of the %d capped rows (%.1f%%)"
      % ((flagged & on_ceiling).sum(), on_ceiling.sum(),
         100 * (flagged & on_ceiling).sum() / on_ceiling.sum()))
print("  and wrongly flags %d of the %d ordinary rows (%.2f%%)"
      % ((flagged & ~on_ceiling).sum(), (~on_ceiling).sum(),
         100 * (flagged & ~on_ceiling).sum() / (~on_ceiling).sum()))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.4))

left.hist(chance_capped[~on_ceiling], bins=40, color="#b0b0b0", label="not at the ceiling")
left.hist(chance_capped[on_ceiling], bins=40, color="#c0392b", alpha=0.85,
          label="at the ceiling")
left.set_yscale("log")
left.set_xlabel("the gate's estimated probability that the row is capped")
left.set_ylabel("test rows (log scale)")
left.set_title("almost every ordinary row is cleared; the capped ones spread out")
left.legend(fontsize=9)

labels = ["all rows", "ordinary rows", "capped rows"]
one = [score(scored["error"].to_numpy()), score(scored["error"].to_numpy()[~on_ceiling]),
       score(scored["error"].to_numpy()[on_ceiling])]
two = [score(combined_error), score(combined_error[~on_ceiling]),
       score(combined_error[on_ceiling])]
positions = np.arange(3)
right.bar(positions - 0.19, one, 0.36, color="#2c5f9e", label="one model")
right.bar(positions + 0.19, two, 0.36, color="#e08a1e", label="two-part model")
for x, (a, b) in enumerate(zip(one, two)):
    right.text(x - 0.19, a + 0.012, "%.4f" % a, ha="center", fontsize=8.5, color="#2c5f9e")
    right.text(x + 0.19, b + 0.012, "%.4f" % b, ha="center", fontsize=8.5, color="#e08a1e")
right.set_xticks(positions)
right.set_xticklabels(labels)
right.set_ylim(0, 1.15)
right.set_ylabel("RMSE on the test set")
right.set_title("and buys almost nothing in RMSE")
right.legend(fontsize=9)

fig.suptitle("E23 - the useful output of the two-part model is not its error", y=1.02)
fig.tight_layout()
plt.show()

**On RMSE the two-part model is a wash: 0.4588 against 0.4574 overall.** It is slightly better on the
ordinary rows (0.4219 against 0.4256) and slightly worse on the capped ones (0.9592 against 0.9068).

**The small gain on ordinary rows is the part that makes sense.** The value model never saw a capped row,
so it was not pulled towards 5.0 by 580 rows whose real values were higher - and it predicts the rows it
*can* be right about slightly better. That is the same effect the chapter's failure lab measured from the
other direction.

**The loss on capped rows makes sense too.** Blending towards exactly 5.00001 is the best possible
prediction of the *recorded* value, and a very poor prediction of the real one. The single model's habit
of over-shooting the ceiling was accidentally closer to the truth.

**And then the left panel, which is the actual result.**

**AUC 0.9642 - but read the histogram rather than the number, because the two errors are not symmetric.**
The grey mass is almost entirely piled at zero: 96.6% of ordinary rows are given less than a 10% chance of
being capped. The red rows are spread right across the range, with the bulk of them high.

**Flagging at 0.5 catches 104 of the 181 capped rows - 57% - while wrongly flagging 31 of 3,947 ordinary
ones, under 1%.** So it is a **high-precision, moderate-recall** flag: when it fires you should believe
it, and when it stays silent you have learnt less than you would like.

**That is still worth more than the 0.0014 RMSE it cost.** The model cannot tell you what those block
groups are worth, but for more than half of them it can tell you not to trust the number - and it almost
never cries wolf.

> **A prediction with a trustworthy "do not rely on this one" flag is a more useful product than a
> marginally better average with no flag** - and where the threshold goes is a decision about which of the
> two mistakes costs more, not something the AUC decides for you.

**The general move, which recurs throughout applied work:** when part of a problem is unanswerable, stop
trying to answer it and build something that identifies it instead. The unanswerable part does not go
away, but it stops being invisible.